# Retrieval Pipeline Diagnostics

## 1. Goal and objectives

Dense retrieval embeds a question and finds chunks with nearby vectors; it is useful when wording differs but meaning is similar. Sparse retrieval uses BM25 keyword matching; it is especially strong for exact terminology, acronyms, identifiers, and rare words. A production RAG system often benefits from both signals, but this notebook tests them separately?hybrid fusion and reranking are intentionally out of scope.

This diagnostic walkthrough verifies the shared query contract, embedding validity, Qdrant collection shape and payloads, the serialized BM25 corpus-to-chunk mapping, filters, and representative searches. Successful indexing means vector dimensions match, Qdrant points have usable text and unique chunk IDs, BM25 positions map one-to-one to complete chunk records, and relevant queries return inspectable results.

## 2. Setup

Run this notebook from the repository root. The setup also locates the root when Jupyter starts in `notebooks/`. It loads `.env` without printing secrets, fixes NumPy's random seed, configures readable logs, and reports?not installs?the key dependency versions.

In [1]:
from __future__ import annotations

import importlib.metadata as metadata
import logging
import os
import sys
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "processing").is_dir() and (REPO_ROOT.parent / "processing").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "processing").is_dir():
    raise RuntimeError("Run this notebook from the RAG-AI_Reasearch_Papers repository")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from processing.embedder import Embedder
from retrieval import DenseRetriever, QueryProcessor, SparseRetriever

packages = ["qdrant-client", "sentence-transformers", "rank-bm25", "numpy"]
versions = {"python": sys.version.split()[0]}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"
display(pd.DataFrame(versions.items(), columns=["dependency", "version"]))
print("Repository:", REPO_ROOT)

D:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


INFO datasets: PyTorch version 2.3.1 available.


,dependency,version
0,python,3.11.9
1,qdrant-client,1.10.1
2,sentence-transformers,3.0.1
3,rank-bm25,0.2.2
4,numpy,1.26.4


Repository: D:\data science\project\RAG-AI_Reasearch_Papers


## 3. Load project configuration

Configuration comes from environment variables with the same defaults used by the processing layer. `BM25_INDEX_PATH` is optional; when the canonical artifact is absent, this notebook selects the existing local-pipeline artifact and says so explicitly.

In [2]:
def env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    return default if raw is None else raw.strip().casefold() in {"1", "true", "yes", "on"}

embedding_model = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
embedding_dimension = int(os.getenv("EMBEDDING_DIM", "384"))
qdrant_host = os.getenv("QDRANT_HOST", "localhost")
qdrant_port = int(os.getenv("QDRANT_PORT", "6333"))
qdrant_collection = os.getenv("QDRANT_COLLECTION", "ai_papers")
dense_top_k = int(os.getenv("DENSE_TOP_K", "10"))
sparse_top_k = int(os.getenv("SPARSE_TOP_K", "10"))
notebook_top_k = int(os.getenv("NOTEBOOK_TOP_K", "5"))
query_expansion_enabled = env_bool("QUERY_EXPANSION_ENABLED", False)

configured_bm25 = Path(os.getenv("BM25_INDEX_PATH", "data/processed/bm25_index.pkl"))
if not configured_bm25.is_absolute():
    configured_bm25 = REPO_ROOT / configured_bm25
fallback_bm25 = REPO_ROOT / "data/local_pipeline_test/processed/bm25_index.pkl"
bm25_index_path = configured_bm25 if configured_bm25.is_file() else fallback_bm25
if bm25_index_path == fallback_bm25 and bm25_index_path.is_file():
    print(f"Canonical BM25 artifact absent; using local diagnostic artifact: {bm25_index_path}")

safe_config = {
    "embedding_model": embedding_model,
    "embedding_dimension": embedding_dimension,
    "qdrant_host": qdrant_host,
    "qdrant_port": qdrant_port,
    "qdrant_collection": qdrant_collection,
    "bm25_index_path": str(bm25_index_path),
    "dense_top_k": dense_top_k,
    "sparse_top_k": sparse_top_k,
    "notebook_top_k": notebook_top_k,
    "query_expansion_enabled": query_expansion_enabled,
}
display(pd.DataFrame(safe_config.items(), columns=["setting", "value"]))

Canonical BM25 artifact absent; using local diagnostic artifact: D:\data science\project\RAG-AI_Reasearch_Papers\data\local_pipeline_test\processed\bm25_index.pkl


,setting,value
0,embedding_model,all-MiniLM-L6-v2
1,embedding_dimension,384
2,qdrant_host,localhost
3,qdrant_port,6333
4,qdrant_collection,ai_papers
5,bm25_index_path,D:\data science\project\RAG-AI_Reasearch_Paper...
6,dense_top_k,50
7,sparse_top_k,50
8,notebook_top_k,5
9,query_expansion_enabled,False


## 4. Initialize and validate the production embedder

A valid query embedding is one-dimensional after selecting the single row, finite, non-empty, and equal to the configured dimension. Model loading can require a previously cached model or network access; a failure is printed with its traceback so later BM25 diagnostics can still run.

In [3]:
embedder = None
query_vector = None
try:
    embedder = Embedder(model_name=embedding_model)
    embedding_batch = embedder.encode_texts(["What is retrieval-augmented generation?"])
    query_vector = np.asarray(embedding_batch[0])
    embedding_diagnostics = {
        "batch_shape": embedding_batch.shape,
        "vector_dimension": query_vector.size,
        "dtype": str(query_vector.dtype),
        "first_values": query_vector[:8].tolist(),
        "contains_nan": bool(np.isnan(query_vector).any()),
        "contains_infinite": bool(np.isinf(query_vector).any()),
        "l2_norm": float(np.linalg.norm(query_vector)),
    }
    display(pd.Series(embedding_diagnostics, name="value"))
    assert embedding_batch.ndim == 2 and embedding_batch.shape[0] == 1
    assert query_vector.size == embedding_dimension
    assert np.isfinite(query_vector).all()
    assert np.linalg.norm(query_vector) > 0
except Exception:
    print("EMBEDDER DIAGNOSTIC FAILED. Check model availability and EMBEDDING_MODEL/DIM.")
    traceback.print_exc()

INFO sentence_transformers.SentenceTransformer: Use pytorch device_name: cpu


INFO sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: all-MiniLM-L6-v2


batch_shape                                                   (1, 384)
vector_dimension                                                   384
dtype                                                          float64
first_values         [-0.11101916432380676, -0.026313403621315956, ...
contains_nan                                                     False
contains_infinite                                                False
l2_norm                                                            1.0
Name: value, dtype: object

## 5. Check Qdrant connection and collection

The health report distinguishes a stopped server, a missing or empty collection, a dimension mismatch, and a non-cosine distance metric. Full exception details remain visible.

In [4]:
qdrant_client = None
dense_retriever = None
dense_health = None
try:
    from qdrant_client import QdrantClient

    qdrant_client = QdrantClient(url=f"http://{qdrant_host}:{qdrant_port}", timeout=5)
    collections = qdrant_client.get_collections().collections
    collection_names = [item.name for item in collections]
    print("Connection status: reachable")
    print("Existing collections:", collection_names)

    if embedder is not None:
        dense_retriever = DenseRetriever(
            qdrant_client, embedder, qdrant_collection, default_top_k=notebook_top_k
        )
        dense_health = dense_retriever.health_check()
    else:
        dense_health = {
            "connected": True,
            "collection_exists": qdrant_collection in collection_names,
            "collection_name": qdrant_collection,
            "embedder_dimension": None,
            "dimension_match": None,
            "note": "Embedder unavailable; full health check deferred",
        }
    display(pd.Series(dense_health, name="value"))

    if not dense_health.get("collection_exists"):
        print("DIAGNOSTIC: expected collection does not exist; run the indexing pipeline.")
    elif dense_health.get("points_count") == 0:
        print("DIAGNOSTIC: collection exists but is empty; index chunks before retrieval.")
    if dense_health.get("dimension_match") is False:
        print("DIAGNOSTIC: collection and embedder dimensions differ; rebuild with one model.")
    if dense_health.get("distance") and str(dense_health["distance"]).casefold() != "cosine":
        print("DIAGNOSTIC: collection distance is not cosine; recreate it with cosine distance.")
except Exception:
    print("QDRANT DIAGNOSTIC FAILED. Confirm the server URL and that Qdrant is running.")
    traceback.print_exc()

QDRANT DIAGNOSTIC FAILED. Confirm the server URL and that Qdrant is running.


Traceback (most recent call last):
  File "D:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\httpx\_transports\default.py", line 69, in map_httpcore_exceptions
    yield
  File "D:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\httpx\_transports\default.py", line 233, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\httpcore\_sync\connection_pool.py", line 256, in handle_request
    raise exc from None
  File "D:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\httpcore\_sync\connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\data science\project\RAG-AI_Reasearch_Papers\venv\Lib\site-packages\httpcore\_sync\connection.py", line 101, in handle_request
    raise exc
  File "D:\data science\project\RAG-AI_Re

## 6. Inspect sample Qdrant points

This bounded scroll checks point IDs, text availability, payload coherence, unique sample chunk IDs, and stored vector dimensions.

In [5]:
qdrant_sample = []
if qdrant_client is not None and dense_health and dense_health.get("collection_exists"):
    try:
        points, _ = qdrant_client.scroll(
            collection_name=qdrant_collection,
            limit=5,
            with_payload=True,
            with_vectors=True,
        )
        for point in points:
            payload = dict(point.payload or {})
            metadata_payload = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else {}
            vector = point.vector
            vector_dimension = len(vector) if isinstance(vector, list) else None
            qdrant_sample.append({
                "point_id": str(point.id),
                "chunk_id": payload.get("chunk_id"),
                "paper_id": payload.get("paper_id", metadata_payload.get("paper_id")),
                "title": payload.get("title", metadata_payload.get("title")),
                "section": payload.get("section", metadata_payload.get("section")),
                "text_preview": str(payload.get("text", ""))[:120],
                "vector_available": vector is not None,
                "vector_dimension": vector_dimension,
                "payload_keys": sorted(payload),
            })
        sample_frame = pd.DataFrame(qdrant_sample)
        display(sample_frame)
        assert all(row["point_id"] for row in qdrant_sample)
        assert all(row["text_preview"] for row in qdrant_sample)
        chunk_ids = [row["chunk_id"] for row in qdrant_sample]
        assert all(chunk_ids) and len(chunk_ids) == len(set(chunk_ids))
        assert all(
            row["vector_dimension"] in (None, embedding_dimension)
            for row in qdrant_sample
        )
    except Exception:
        print("SAMPLE POINT INSPECTION FAILED; collection details are preserved above.")
        traceback.print_exc()
else:
    print("Skipped: Qdrant is unavailable or the configured collection does not exist.")

Skipped: Qdrant is unavailable or the configured collection does not exist.


## 7. Load and validate the trusted BM25 artifact

**Security:** this artifact is a Python pickle. Pickles can execute code during loading. Only load trusted, local files generated by this project?never an arbitrary downloaded pickle.

In [6]:
sparse_retriever = None
sparse_health = None
try:
    sparse_retriever = SparseRetriever(bm25_index_path, default_top_k=notebook_top_k)
    sparse_health = sparse_retriever.health_check()
    display(pd.Series(sparse_health, name="value"))
    assert sparse_health["document_count"] == sparse_health["chunk_mapping_count"]
    assert sparse_health["mapping_valid"]

    corpus_rows = []
    for position, (tokens, chunk) in enumerate(
        zip(sparse_retriever.documents[:5], sparse_retriever.chunks[:5], strict=True)
    ):
        metadata_payload = chunk.get("metadata") if isinstance(chunk.get("metadata"), dict) else {}
        corpus_rows.append({
            "corpus_position": position,
            "chunk_id": chunk.get("chunk_id"),
            "paper_id": chunk.get("paper_id", metadata_payload.get("paper_id")),
            "section": chunk.get("section", metadata_payload.get("section")),
            "token_preview": tokens[:15],
            "text_preview": str(chunk.get("text", ""))[:120],
        })
    display(pd.DataFrame(corpus_rows))
except Exception:
    print("BM25 DIAGNOSTIC FAILED. Verify BM25_INDEX_PATH and rebuild the sparse index.")
    traceback.print_exc()

loaded                                                               True
index_path              D:\data science\project\RAG-AI_Reasearch_Paper...
document_count                                                        173
chunk_mapping_count                                                   173
metadata_count                                                        173
mapping_valid                                                        True
preprocessing_config    {'tokenizer': 'word_v1', 'lowercase': True, 's...
version                                                               1.0
created_at                                                           None
Name: value, dtype: object

,corpus_position,chunk_id,paper_id,section,token_preview,text_preview
0,0,fdfa3141-1f9b-5fb0-97f9-e5d2e722d3a5,2012.13026v1,abstract,"[recently, deep, reinforcement, learning, drl,...","Recently, deep reinforcement learning (DRL)-ba..."
1,1,126dfb0e-33ea-56da-8bd8-4ca103366ab7,2012.13026v1,introduction,"[nowadays, the, rapid, development, of, artifi...","Nowadays, the rapid development of artiﬁcial i..."
2,2,f0ce7094-ecce-5caa-b2ad-1cf90e3c530b,2012.13026v1,introduction,"[lessons, learnt, from, training, an, effectiv...",lessons learnt from training an effective DRL ...
3,3,d2e6ba49-eb17-5b37-8f56-6d38adff5012,2012.13026v1,introduction,"[flow, solver, simulator, to, model, the, tran...",ﬂow solver (simulator) to model the transition...
4,4,c3f47a8e-d0fa-5be1-bffd-5c29ef76959a,2012.13026v1,introduction,"[to, a, successful, state, in, as, few, number...",to a successful state in as few number of step...


## 8. Demonstrate query processing

Cleaning preserves technical spelling for dense retrieval. Sparse tokens deliberately follow the BM25 artifact's preprocessing contract; a legacy artifact may therefore split hyphenated terms while a new v2 artifact retains them.

In [7]:
preprocessing_config = (
    sparse_retriever.preprocessing_config if sparse_retriever is not None else None
)
query_processor = QueryProcessor(
    enable_expansion=query_expansion_enabled,
    preprocessing_config=preprocessing_config,
)
demo_queries = [
    "What is retrieval-augmented generation?",
    "How does multi-head attention work?",
    "Which datasets are used to evaluate RAG systems?",
    "Compare BM25 and dense retrieval.",
    "What is Q-learning?",
]
processed_rows = []
for query in demo_queries:
    processed = query_processor.process(query)
    processed_rows.append({
        "original_query": processed.original_query,
        "cleaned_query": processed.cleaned_query,
        "dense_query": processed.dense_query,
        "sparse_tokens": processed.sparse_tokens,
        "expanded_query": processed.expanded_query,
    })
display(pd.DataFrame(processed_rows))

technical_terms = "GPT-4 C++ F1-score LLaMA-3 Q-learning"
technical = query_processor.process(technical_terms)
print("Cleaned technical terms:", technical.cleaned_query)
print("Sparse technical tokens:", technical.sparse_tokens)
for term in ["GPT-4", "C++", "F1-score", "LLaMA-3", "Q-learning"]:
    assert term in technical.cleaned_query

,original_query,cleaned_query,dense_query,sparse_tokens,expanded_query
0,What is retrieval-augmented generation?,What is retrieval-augmented generation?,What is retrieval-augmented generation?,"[what, is, retrieval, augmented, generation]",None
1,How does multi-head attention work?,How does multi-head attention work?,How does multi-head attention work?,"[how, does, multi, head, attention, work]",None
2,Which datasets are used to evaluate RAG systems?,Which datasets are used to evaluate RAG systems?,Which datasets are used to evaluate RAG systems?,"[which, datasets, are, used, to, evaluate, rag...",None
3,Compare BM25 and dense retrieval.,Compare BM25 and dense retrieval.,Compare BM25 and dense retrieval.,"[compare, bm25, and, dense, retrieval]",None
4,What is Q-learning?,What is Q-learning?,What is Q-learning?,"[what, is, q, learning]",None


Cleaned technical terms: GPT-4 C++ F1-score LLaMA-3 Q-learning
Sparse technical tokens: ['gpt', '4', 'c', 'f1', 'score', 'llama', '3', 'q', 'learning']


## 9. Run dense retrieval

This uses the cleaned/optionally expanded dense query and returns the common result model. The example also demonstrates an exact metadata filter; adjust it to fields present in your collection.

In [8]:
dense_results = []
if dense_retriever is not None and dense_health and dense_health.get("collection_exists"):
    try:
        processed = query_processor.process("What is retrieval-augmented generation?")
        dense_results = dense_retriever.search(processed.dense_query, top_k=notebook_top_k)
        display(pd.DataFrame([
            {
                "rank": rank,
                "chunk_id": result.chunk_id,
                "score": result.score,
                "paper_id": result.paper_id,
                "title": result.title,
                "section": result.section,
                "text_preview": result.text[:160],
            }
            for rank, result in enumerate(dense_results, 1)
        ]))
        assert len(dense_results) <= notebook_top_k
        assert all(result.source == "dense" for result in dense_results)
        assert all(np.isfinite(result.score) for result in dense_results)
    except Exception:
        print("DENSE SEARCH FAILED. Inspect collection health, dimensions, and payload text.")
        traceback.print_exc()
else:
    print("Skipped: dense dependencies or indexed collection are unavailable.")

Skipped: dense dependencies or indexed collection are unavailable.


## 10. Run sparse retrieval and metadata filtering

BM25 scores are raw relevance scores, not probabilities. The first table is unfiltered; the second uses a year range only when the artifact contains numeric years.

In [9]:
sparse_results = []
if sparse_retriever is not None:
    processed = query_processor.process("Compare BM25 and dense retrieval.")
    sparse_results = sparse_retriever.search(
        processed.sparse_tokens, top_k=notebook_top_k
    )
    display(pd.DataFrame([
        {
            "rank": rank,
            "chunk_id": result.chunk_id,
            "score": result.score,
            "paper_id": result.paper_id,
            "title": result.title,
            "year": result.year,
            "section": result.section,
            "text_preview": result.text[:160],
        }
        for rank, result in enumerate(sparse_results, 1)
    ]))
    assert len(sparse_results) <= notebook_top_k
    assert all(result.source == "sparse" for result in sparse_results)
    assert all(np.isfinite(result.score) for result in sparse_results)

    numeric_years = [
        chunk.get("metadata", {}).get("year")
        for chunk in sparse_retriever.chunks
        if isinstance(chunk.get("metadata"), dict)
        and isinstance(chunk.get("metadata", {}).get("year"), int)
    ]
    if numeric_years:
        cutoff = max(numeric_years) - 2
        filtered_results = sparse_retriever.search(
            processed.sparse_tokens,
            top_k=notebook_top_k,
            filters={"year": {"gte": cutoff}},
        )
        print(f"Results with year >= {cutoff}: {len(filtered_results)}")
        assert all(result.year is not None and result.year >= cutoff for result in filtered_results)
else:
    print("Skipped: BM25 artifact is unavailable or invalid.")

,rank,chunk_id,score,paper_id,title,year,section,text_preview
0,1,bc92b6da-dffe-5067-b91e-82a8b4931515,9.172180,2012.13391v2,"I like fish, especially dolphins: Addressing C...",2020,experiments,.7 / 84.3 Electra DECODE 93.17 81.19 80.76 87....
1,2,35302bdc-bc43-5cd9-b216-3931b74f1f93,5.819602,2012.13391v2,"I like fish, especially dolphins: Addressing C...",2020,abstract,To quantify how well natural language un- ders...
2,3,2e056bb6-f7d7-57a6-a0a7-0b5ff7261b1c,5.324247,2012.13391v2,"I like fish, especially dolphins: Addressing C...",2020,experiments,can have very different performance on OOD hum...
3,4,2adb4a0e-b910-5b6a-82b8-bee3b67eaa09,4.937107,2012.13391v2,"I like fish, especially dolphins: Addressing C...",2020,experiments,on all evaluation sets in both the unstructure...
4,5,84cd0f74-7941-538b-9eb2-bf88696608dc,4.917408,2012.13391v2,"I like fish, especially dolphins: Addressing C...",2020,experiments,"Re-ranking Given a contradiction detector, an ..."


Results with year >= 2018: 5


## 11. Compare backend outputs (no fusion)

This is a diagnostic side-by-side comparison only. Scores are intentionally not combined because cosine and BM25 scores have different scales. Hybrid fusion and reranking belong in future production modules.

In [10]:
comparison_rows = []
for backend, results in [("dense", dense_results), ("sparse", sparse_results)]:
    for rank, result in enumerate(results, 1):
        comparison_rows.append({
            "backend": backend,
            "rank": rank,
            "chunk_id": result.chunk_id,
            "raw_score": result.score,
            "paper_id": result.paper_id,
            "section": result.section,
            "text_preview": result.text[:120],
        })
comparison = pd.DataFrame(comparison_rows)
display(comparison)
if dense_results and sparse_results:
    overlap = {result.chunk_id for result in dense_results} & {
        result.chunk_id for result in sparse_results
    }
    print(f"Top-{notebook_top_k} chunk overlap: {len(overlap)} -> {sorted(overlap)}")
else:
    print("A two-backend comparison requires both successful searches.")

,backend,rank,chunk_id,raw_score,paper_id,section,text_preview
0,sparse,1,bc92b6da-dffe-5067-b91e-82a8b4931515,9.172180,2012.13391v2,experiments,.7 / 84.3 Electra DECODE 93.17 81.19 80.76 87....
1,sparse,2,35302bdc-bc43-5cd9-b216-3931b74f1f93,5.819602,2012.13391v2,abstract,To quantify how well natural language un- ders...
2,sparse,3,2e056bb6-f7d7-57a6-a0a7-0b5ff7261b1c,5.324247,2012.13391v2,experiments,can have very different performance on OOD hum...
3,sparse,4,2adb4a0e-b910-5b6a-82b8-bee3b67eaa09,4.937107,2012.13391v2,experiments,on all evaluation sets in both the unstructure...
4,sparse,5,84cd0f74-7941-538b-9eb2-bf88696608dc,4.917408,2012.13391v2,experiments,"Re-ranking Given a contradiction detector, an ..."


A two-backend comparison requires both successful searches.


## 12. Checks and next steps

Use the health reports and bounded samples above as the handoff record. Before implementing fusion or reranking:

1. Resolve any Qdrant connectivity, empty-collection, distance, or dimension diagnostics.
2. Rebuild legacy BM25 artifacts with the v2 format so the stored tokenized corpus and technical-term tokenizer are explicit.
3. Confirm representative dense and sparse results contain coherent paper metadata and useful text.
4. Add evaluation queries with known relevant chunks, then measure recall separately for each backend.

The production modules under `retrieval/` remain the source of truth; this notebook contains diagnostics and examples, not duplicate retrieval implementations.